# FF-HEDM from raw CBF — auto-seed calibration + c-omp reconstruction

End-to-end far-field HEDM on a real dataset, using the **new MIDAS Python
stack**:

1. **Calibrate** the detector from a LaB6 (or any) calibrant with
   `midas-calibrate-v2`'s **auto-seed** one-shot path — no prior parameter
   file needed. Beam center and sample-to-detector distance are found from the
   rings themselves.
2. **Build** the FF parameter file from the refined geometry.
3. **Reconstruct** grains with `midas-pipeline` in FF mode using the
   **`c-omp` indexer backend** (the bundled unified C `midas_indexer`).
4. **Inspect** the recovered grains.

**Worked example:** a Ni sample on a Varex XRD-4343CT (2880x2880, 150 um pixel,
CBF), 1800 omega frames, calibrated against LaB6.

> ### ⚠️ Read detector images with the MIDAS reader, never `fabio`
> MIDAS works in a transposed + double-flipped frame
> (`pixels.reshape(nrows, ncols).T[::-1, ::-1]`). `fabio` returns the *raw*
> frame, which is a Y↔Z transpose + both-axes flip of the MIDAS frame — it
> gives a mirrored beam center and **mirrored tilts/distortion that are invalid
> for the pipeline**. Always read via `midas_zipper._read_cbf.read_cbf`
> (`midas-calibrate-v2` >= 0.2.2 already does this internally).


## Install — start from scratch

`midas-suite` is a meta-package that pulls the entire MIDAS Python stack
(calibration, zipper, peakfit, transforms, index **+ the c-omp `midas_indexer`
binary**, fit-grain, process-grains, the FF/PF orchestrators) in one command.

The `c-omp` indexer is a compiled OpenMP binary built at install time, so you
need a C toolchain + OpenMP (Linux: `gcc`; macOS: `brew install libomp`).


In [ ]:
# Run once in a fresh environment (uncomment to execute):
# %pip install "midas-suite[ff,plots]"     # FF-HEDM stack + orchestrator + matplotlib (for the figures below)
# Verify the c-omp indexer binary built:
from midas_index import backend_c
print("c-omp indexer available:", backend_c.available(), "->", backend_c.binary_path())
import midas_suite; print("installed:", len(midas_suite.installed()), "midas packages")

## 0. Configuration — edit this cell for your dataset

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # diplib/OpenMP on the seed step
from pathlib import Path

# ----- paths -----------------------------------------------------------------
WORK         = Path("/gdata/dm/MPE/OrthrosJr/analysis/sharma_work/indrajeet_ni_ff")
LAB6_DIR     = WORK / "LAB6/deg0_exp0_3/Varex_1"   # dir of calibrant CBF frames (ω = 0°)
LAB6_DIR_180 = WORK / "LAB6/deg180_exp0_3/Varex_1" # opposing 180° rotation — averaged with the 0° fit
                                                   # to cancel the sample beam-direction offset baked into Lsd
NI_RAW       = WORK / "Ni_ht/Varex_1"              # dir of sample CBF frames
CALIB_OUT    = WORK / "nb_calib"
PS_V2        = WORK / "nb_ps_ni_v2.txt"
RECON_OUT    = WORK / "nb_ni_recon"

# ----- calibrant (LaB6) ------------------------------------------------------
CAL_A, CAL_SG = 4.1569, 221         # LaB6 cubic, Pm-3m
WAVELENGTH    = 0.2066              # Angstrom  (= sample wavelength)
PX            = 150.0               # µm
NPIX          = 2880
LSD_GUESS     = 960000.0            # µm — rough nominal distance (order of magnitude)

# ----- sample / scan ---------------------------------------------------------
SAMPLE_LATTICE = (3.6, 3.6, 3.6, 90.0, 90.0, 90.0)   # Ni cubic Fm-3m: a, b, c, α, β, γ
SAMPLE_SG      = 225
OMEGA_START    = 180.0
OMEGA_STEP     = -0.2
OMEGA_RANGE    = (-180.0, 180.0)
NUM_FILES_PER_SCAN = 1800           # 1 frame per CBF -> number of CBF files
                                    # (CRITICAL for per-frame CBF; default 1
                                    #  reads only the first file!)
FILE_STEM      = "frame_%I.cbf_00001_Varex_1"
FILE_EXT       = ".cbf"
FILE_PADDING   = 5
START_NR       = 1
END_NR         = 1800
SAT_THRESHOLD  = 70000              # detector saturation cutoff

# Peak intensity thresholds per ring (rings 1..N). Hand-picked from the sample
# rings' intensity histograms — tune per material/exposure.
RING_THRESH    = {1: 150, 2: 150, 3: 150, 4: 150, 5: 150}
RING_TO_INDEX  = 2                  # which ring the indexer pivots on

# Sample / illumination geometry (µm). V_SAMPLE_UM3 is the *physical*
# illuminated volume — Stage 8.5 of the v4 pipeline uses it as the
# intensity-conservation budget.
V_SAMPLE_UM3   = 1.8e7              # 300 × 300 × 200 µm³ (illuminated)
BEAM_THICKNESS = 200                # µm
R_SAMPLE       = 2000               # µm (worst-case sample radius)
H_BEAM         = 2000               # µm (worst-case sample height)

# ----- compute ---------------------------------------------------------------
N_CPUS          = 64
INDEXER_BACKEND = "c-omp"           # bundled unified C midas_indexer
REFINE_BACKEND  = "c-omp"           # bundled unified C midas_fitgrain (FF refines
                                    # position; PF fixes it). "python" = PyTorch refiner.

## 1. Calibrate from the LaB6 rings (auto-seed, one-shot)
We median the calibrant frames (stationary powder rings ⇒ median removes zingers/hot pixels), then call `calibrate(...)`, which auto-seeds beam center + Lsd from the rings and LM-refines geometry + distortion.

In [ ]:
import numpy as np
from midas_zipper._read_cbf import read_cbf      # MIDAS reader (NOT fabio)
import glob
from midas_calibrate_v2 import calibrate

def median_calibrant(raw_dir):
    files = sorted(glob.glob(str(Path(raw_dir) / "*.cbf")))
    stack = np.stack([read_cbf(f, check_md5=False)[1].astype(np.float32)
                      for f in files])
    print(f"  {Path(raw_dir).name}: median over {len(files)} frames -> {stack.shape}")
    return np.median(stack, axis=0).astype(np.float64)

def calibrate_one(img, tag):
    return calibrate(
        img, wavelength=WAVELENGTH, pxY=PX,
        calibrant={"a": CAL_A, "sg": CAL_SG},
        output_dir=str(CALIB_OUT / tag),
        initial_Lsd=LSD_GUESS, verbose=True,
    )

# Calibrate BOTH 0° and 180° (the opposing rotation cancels the
# sample beam-direction offset that a single-image fit bakes into Lsd —
# typically ~1000 µm Lsd / 1000 µε hydrostatic if you only use one image).
print("[cal] computing medians...")
cal_img_0   = median_calibrant(LAB6_DIR)
cal_img_180 = median_calibrant(LAB6_DIR_180)

print("\n[cal] STAGE 1 / 0° image:")
res0 = calibrate_one(cal_img_0, "deg0")
print("\n[cal] STAGE 2 / 180° image:")
res180 = calibrate_one(cal_img_180, "deg180")

print(f"\n[cal] deg0   : Lsd={res0.Lsd/1000:.3f}mm  BC=({res0.BC_y:.3f},{res0.BC_z:.3f})  ty={res0.ty:+.4f} tz={res0.tz:+.4f}  strain={res0.post_residual_strain_uE:.1f}μϵ")
print(f"[cal] deg180 : Lsd={res180.Lsd/1000:.3f}mm  BC=({res180.BC_y:.3f},{res180.BC_z:.3f})  ty={res180.ty:+.4f} tz={res180.tz:+.4f}  strain={res180.post_residual_strain_uE:.1f}μϵ")

# Average the two — keeps everything but Lsd; Lsd is where the 0/180
# offset matters. Build a thin "averaged result" view by copying res0
# and replacing the geometry fields with the mean.
import copy
res = copy.copy(res0)
res.Lsd  = 0.5 * (res0.Lsd  + res180.Lsd)
res.BC_y = 0.5 * (res0.BC_y + res180.BC_y)
res.BC_z = 0.5 * (res0.BC_z + res180.BC_z)
res.ty   = 0.5 * (res0.ty   + res180.ty)
res.tz   = 0.5 * (res0.tz   + res180.tz)
# Distortion: average the v2 dict in place
for k in res0.distortion:
    res.distortion[k] = 0.5 * (res0.distortion[k] + res180.distortion[k])
# Drop residual_corr_map (per-image; not meaningful after averaging).
res.residual_corr_map = None
res.residual_corr_bin_path = None

print(f"\n[cal] AVERAGED : Lsd={res.Lsd/1000:.3f}mm  BC=({res.BC_y:.3f},{res.BC_z:.3f})  ty={res.ty:+.4f} tz={res.tz:+.4f}")
print(f"        (deg0 / deg180 Lsd spread: {(res0.Lsd-res180.Lsd):.0f}μm = {(res0.Lsd-res180.Lsd)/res.Lsd*1e6:+.0f}μϵ — captures the sample beam-direction offset)")


### Sanity check — predicted rings must sit on the measured signal
Uses `midas_hkls` (space-group extinction rules) to predict the allowed LaB6 rings at the refined geometry and overlays them on the median image.

In [ ]:
import math, matplotlib.pyplot as plt
from midas_hkls import SpaceGroup, Lattice, generate_hkls

refs = generate_hkls(SpaceGroup.from_number(CAL_SG),
                     Lattice(a=CAL_A, b=CAL_A, c=CAL_A,
                             alpha=90., beta=90., gamma=90.),
                     wavelength_A=res.wavelength_A, two_theta_max_deg=18.0)
ring_R = sorted(res.Lsd * math.tan(math.radians(r.two_theta_deg)) / res.pxY
                for r in refs)

fig, ax = plt.subplots(figsize=(9, 9))
pos = cal_img_0[cal_img_0 > 0]
ax.imshow(np.log1p(cal_img_0.clip(min=1)), origin="lower", cmap="gray",
          vmin=np.log1p(np.percentile(pos, 60)),
          vmax=np.log1p(np.percentile(pos, 99.9)))
th = np.linspace(0, 2*np.pi, 500)
for r in ring_R:
    ax.plot(res.BC_y + r*np.cos(th), res.BC_z + r*np.sin(th),
            "-", color="lime", lw=0.7, alpha=0.85)
ax.plot(res.BC_y, res.BC_z, "ro", ms=7, mec="yellow")
ax.set_title(f"LaB6 median + predicted rings @ averaged 0°/180° geometry")
ax.set_xlim(0, res.NrPixelsY); ax.set_ylim(0, res.NrPixelsZ)
plt.tight_layout(); plt.show()


## 2. Build the FF parameter file

Carry the **full** detector model from the powder calibration straight into
HEDM — refined geometry (`Lsd`, beam center, **all** tilts `tx/ty/tz`), the
**complete v2 distortion** (`iso_R2/4/6`, `a1..a6`, `phi1..phi6`), pixel
size, detector shape — into one combined paramstest. The non-geometry
parameters (sample / scan / file naming / peakfit / indexer thresholds)
come from the `overrides=` dict below and the registry defaults; the
geometry / detector keys come from the calibration result `res`.

`midas_params.build_paramstest` is the single API call — it merges
**registry defaults < calibration seeds < user overrides** (highest
precedence) into a validated paramstest. The same registry knows what
fields exist and catches typos / missing-key silent defaults (e.g.
the long-standing "grain-tx returns 0 matched spots because `px` got
silently defaulted to 0" trap).

> Want to **edit values interactively**? Replace this call with the
> `midas_params.notebook.widget(...)` cell below (commented out by
> default — running it in Jupyter Lab/notebook renders an editable
> form and writes the same paramstest on click). The non-interactive
> path stays the canonical reproducible one for nbconvert / CI.

In [ ]:
from midas_params import build_paramstest

# Non-geometry / non-detector parameters. Anything you'd previously have
# pasted into a `ps_*.txt` template lives here as a Python dict — the
# midas-params registry validates types + names, so typos surface inline.
FF_OVERRIDES = dict(
    # Phase
    LatticeParameter = list(SAMPLE_LATTICE),     # [a, b, c, α, β, γ]
    SpaceGroup       = SAMPLE_SG,
    Wavelength       = WAVELENGTH,
    NumPhases        = 1,
    PhaseNr          = 1,

    # Detector hardware NOT carried by the v2 calibration result
    ImTransOpt       = 0,
    # RhoD = corner radius of the detector (µm). MaxRingRad is a registry
    # alias of RhoD — don't double-write.
    RhoD             = ( (NPIX/2) * PX ) * (2 ** 0.5),
    Wedge            = 0,

    # File naming
    Ext              = FILE_EXT,
    Padding          = FILE_PADDING,
    StartNr          = START_NR,
    EndNr            = END_NR,
    FileStem         = FILE_STEM,
    StartFileNrFirstLayer = START_NR,
    NrFilesPerSweep  = NUM_FILES_PER_SCAN,

    # Scan geometry
    OmegaStart       = OMEGA_START,
    OmegaStep        = OMEGA_STEP,
    OmegaRange       = list(OMEGA_RANGE),

    # Sample / illumination
    Vsample          = V_SAMPLE_UM3,
    BeamThickness    = BEAM_THICKNESS,
    GlobalPosition   = 0,
    Rsample          = R_SAMPLE,
    Hbeam            = H_BEAM,

    # Peakfit
    UpperBoundThreshold = SAT_THRESHOLD,
    RingThresh       = [[r, t] for r, t in sorted(RING_THRESH.items())],

    # Indexer
    OverAllRingToIndex = RING_TO_INDEX,
    Completeness     = 0.5,
    MinNrSpots       = 3,
    UseFriedelPairs  = 1,
    StepSizeOrient   = 0.2,
    StepSizePos      = 500,
    MinOmeSpotIDsToIndex = -90,
    MaxOmeSpotIDsToIndex = +90,
    MinEta           = 6,
    Width            = 2000,
    MarginRadial     = 1000,
    MarginEta        = 500,
    MarginOme        = 0.6,
    MarginRadius     = 1000,
    BoxSize          = [-1e6, 1e6, -1e6, 1e6],
    OmeBinSize       = 0.1,
    EtaBinSize       = 0.1,
    MargABC          = 5,
    MargABG          = 5,

    # Sample raw-data location (overrides anything from the seed)
    RawFolder        = str(NI_RAW) + ("/" if not str(NI_RAW).endswith("/") else ""),
    NrPixelsY        = NPIX,
    NrPixelsZ        = NPIX,
)

# Build the paramstest non-interactively:
#   * seed_from=res          → pulls Lsd, BC, tx/ty/tz, pxY/pxZ, NrPixels,
#                              and the v2 distortion harmonics from the
#                              powder calibration result
#   * overrides=FF_OVERRIDES → everything above
#   * validate=True          → registry validators report any typos /
#                              cross-field issues inline before pipeline runs
PS_V2 = build_paramstest(
    out_path   = PS_V2,
    seed_from  = res,
    overrides  = FF_OVERRIDES,
    path       = "ff",
    validate   = True,
)
print(f"wrote {PS_V2}")
print(f"  Lsd={res.Lsd/1000:.3f} mm  BC=({res.BC_y:.2f},{res.BC_z:.2f})  "
      f"tx={res.tx:+.4f} ty={res.ty:+.4f} tz={res.tz:+.4f}")
print(f"  RhoD={FF_OVERRIDES['RhoD']:.0f} µm   wavelength={WAVELENGTH} Å   "
      f"px={(res.pxY + res.pxZ) / 2 if res.pxZ > 0 else res.pxY:.1f} µm")
print("  distortion (v2): "
      + ", ".join(f"{k}={v:+.2e}" for k, v in res.distortion.items()))
print(f"  residual map: {res.residual_corr_bin_path}")

### (optional) edit values interactively

If you're running this notebook live in Jupyter Lab / Jupyter Notebook,
uncomment the cell below. It renders an editable form pre-filled from the
calibration result + `FF_OVERRIDES`; you tweak values, click **Build
paramstest**, and the same `PS_V2` file is (re)written with registry
validation. Skip this cell when running `jupyter nbconvert` — ipywidgets
need a live kernel.

In [ ]:
# Uncomment this block to use the interactive widget (Jupyter Lab / Notebook only).
# Requires: pip install ipywidgets
#
# from midas_params.notebook import widget
# w = widget(out_path=PS_V2, seed_from=res, overrides=FF_OVERRIDES, path="ff")
# w

## 3. Reconstruct with `midas-pipeline` (FF mode, c-omp index + refine)
One command runs the full FF pathway: convert (CBF→`.MIDAS.zip`) → hkl → peakfit → transforms → binning → **index (c-omp)** → **refine (c-omp)** → process-grains → `Grains.csv`. Run it on a multicore node.

Both backends are the bundled unified C binaries (no NLopt dependency): `midas_indexer` and `midas_fitgrain` (`FitUnified`). The refiner refines grain position in FF (validated to ~10µm median vs the legacy `FitPosOrStrainsOMP`) and fixes it to the voxel grid in PF. Set `REFINE_BACKEND='python'` for the differentiable PyTorch refiner instead (GPU/MPS, UQ).

> `--num-files-per-scan` **must** equal the number of single-frame CBF files — the default (1) reads only the first frame.
> Use `--device cpu` if the node has no GPU. The c-omp backends need `midas-index` / `midas-fit-grain` installed with an OpenMP toolchain.


In [ ]:
import subprocess, shlex
cmd = (f"midas-pipeline run --scan-mode ff --indexer-backend {INDEXER_BACKEND} --refine-backend {REFINE_BACKEND} "
       f"--num-files-per-scan {NUM_FILES_PER_SCAN} "
       f"--params {PS_V2} --result {RECON_OUT} --n-cpus {N_CPUS} "
       f"--machine local --device cpu")
print(cmd)
# Streams to the notebook. For very long runs, launch in a terminal with
# `nohup ... > log 2>&1 &` and watch the log instead.
env = {**os.environ, "KMP_DUPLICATE_LIB_OK": "TRUE"}
proc = subprocess.run(shlex.split(cmd), env=env)
print("return code:", proc.returncode)


## 4. Inspect the recovered grains

In [ ]:
import pandas as pd
gfile = RECON_OUT / "LayerNr_1" / "Grains.csv"
# Grains.csv has a multi-line % header; the data header row starts with %ID
# (midas-process-grains) or %GrainID (legacy C ProcessGrains).
lines = gfile.read_text().splitlines()
hdr = next(i for i, l in enumerate(lines)
           if l.startswith("%ID") or l.startswith("%GrainID"))
ngrains = int([l for l in lines if l.startswith("%NumGrains")][0].split()[1])
cols = lines[hdr].lstrip("%").split()
df = pd.read_csv(gfile, sep="\t", skiprows=hdr + 1, names=cols, comment="%")
print(f"NumGrains = {ngrains};  rows parsed = {len(df)}")
print(f"Confidence: median={df.Confidence.median():.3f}, "
      f">0.7: {(df.Confidence > 0.7).sum()} grains")
_show = [c for c in ("ID", "GrainID", "X", "Y", "Z", "Confidence",
                     "GrainRadius", "E11", "E22", "E33", "RMSErrorStrain")
         if c in df.columns]
df[_show].head()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
sc = ax[0].scatter(df.X, df.Y, c=df.Confidence, s=8, cmap="viridis")
ax[0].set(xlabel="X (um)", ylabel="Y (um)", title="grain map (color=confidence)")
ax[0].set_aspect("equal"); fig.colorbar(sc, ax=ax[0])
ax[1].hist(df.Confidence, bins=40, color="steelblue")
ax[1].set(xlabel="completeness/confidence", ylabel="grains", title="confidence")
ax[2].hist(df.GrainRadius, bins=40, color="indianred")
ax[2].set(xlabel="grain radius (um)", ylabel="grains", title="grain size")
plt.tight_layout(); plt.show()


### (optional) Compare against a legacy reconstruction
If you have an older `Grains.csv`, compare grain counts and confidence distributions. Note: a calibration with a wrong beam center (e.g. from a fabio-mirrored read) will shift spot positions and degrade indexing — so a grain-count change after re-calibrating is expected and usually an improvement.

In [ ]:
LEGACY = WORK / "Ni_ht/ff_Al6/LayerNr_1/Grains.csv"
if LEGACY.exists():
    ll = LEGACY.read_text().splitlines()
    legacy_n = int([l for l in ll if l.startswith("%NumGrains")][0].split()[1])
    print(f"legacy NumGrains = {legacy_n};  new NumGrains = {ngrains}")
else:
    print("no legacy Grains.csv found — skipping comparison")


## 5. Refine `tx` from the grains, then re-reconstruct (two-pass)

Powder/calibrant calibration is **structurally blind to `tx`** — the in-plane
detector rotation about the beam. Symmetric Debye–Scherrer rings are invariant
under it, so the geometry from step 1 has `tx = 0`. Single-crystal **grain
spots** are *not* invariant: `tx` rotates each spot's azimuth η by ≈`tx` (with
near-zero effect on the radius), so the recovered grains pin it. (Verified on
real Ni data: `tx`→η is ~1:1, ΔR ≈ 0.)

Standard iterative detector calibration is therefore a **two-pass** flow:

1. Reconstruct with the powder geometry (`tx = 0`) — done above.
2. Refine `tx` from the grains and **re-reconstruct** with the corrected geometry.

The pipeline can do step 2's refine automatically inside an FF run with
`--grain-geometry-run` (the FF-only `grain_geometry` stage, after
`process_grains`). Below we use the standalone `midas-joint-ff-calibrate
grain-tx` tool on the existing layer dir — it's fast (no image re-processing) and
makes the before/after explicit.

In [ ]:
# Refine tx from the grains recovered in step 3 (standalone tool, η-sensitive
# 3D loss). Writes a corrected per-layer paramstest; we then fold the refined
# tx into the master FF param file for the re-run.
import subprocess, shlex, re
LAYER1 = RECON_OUT / "LayerNr_1"
PS_TX_LAYER = LAYER1 / "paramstest_graintx.txt"
# NB: pass the FULL master FF param (PS_V2) -- it carries the omega-scan +
# detector keys (OmegaStep, NrPixels...) the forward model needs. The layer's
# stripped paramstest.txt omits them (OmegaStep would default to 0 -> 0 matches).
cmd = (f"midas-joint-ff-calibrate grain-tx "
       f"--paramstest {PS_V2} --layer-dir {LAYER1} "
       f"--refine tx --kind angular --max-grains 50 --out {PS_TX_LAYER}")
print(cmd)
env = {**os.environ, "KMP_DUPLICATE_LIB_OK": "TRUE"}
subprocess.run(shlex.split(cmd), env=env)

def _tx(p):
    for line in open(p):
        s = line.split('#', 1)[0].strip().rstrip(';').split()
        if s and s[0] == 'tx':
            return float(s[1])
    return 0.0
tx_ref = _tx(PS_TX_LAYER)
print(f"refined tx = {tx_ref:+.6f} deg  (powder pass had tx = 0)")

# Corrected MASTER FF param file for the re-run: PS_V2 with tx set.
txt = PS_V2.read_text()
if re.search(r'(?m)^tx\b', txt):
    txt = re.sub(r'(?m)^tx\b.*$', f'tx {tx_ref:.6f}', txt)
else:
    txt += f"\ntx {tx_ref:.6f}\n"
PS_V2_TX = PS_V2.with_name(PS_V2.stem + "_tx" + PS_V2.suffix)
PS_V2_TX.write_text(txt)
print("wrote corrected master params →", PS_V2_TX)

Re-reconstruct with the corrected `tx`. A fresh result dir gives a clean
side-by-side; peak-fitting is geometry-independent, so on the same dir you could
instead `midas-pipeline resume --from transforms` to reuse step 3's peakfit and
only re-run transforms → binning → index → refine → grains.

In [ ]:
# Pass 2 — re-reconstruct with the tx-corrected geometry (long; same runtime as
# step 3). Comment out if you only want the tx value.
RECON_OUT2 = RECON_OUT.parent / (RECON_OUT.name + "_tx")
cmd2 = (f"midas-pipeline run --scan-mode ff --indexer-backend {INDEXER_BACKEND} "
        f"--refine-backend {REFINE_BACKEND} --num-files-per-scan {NUM_FILES_PER_SCAN} "
        f"--params {PS_V2_TX} --result {RECON_OUT2} --n-cpus {N_CPUS} "
        f"--machine local --device cpu")
print(cmd2)
subprocess.run(shlex.split(cmd2), env=env)
print("return code recorded above")

In [ ]:
# Compare pass 1 (tx=0) vs pass 2 (tx refined): grain count, confidence, strain.
import numpy as np, pandas as pd
def _load_grains(layer):
    g = layer / "Grains.csv"
    lines = g.read_text().splitlines()
    hdr = next(i for i, l in enumerate(lines)
               if l.startswith("%ID") or l.startswith("%GrainID"))
    cols = lines[hdr].lstrip("%").split()
    n = int([l for l in lines if l.startswith("%NumGrains")][0].split()[1])
    df = pd.read_csv(g, sep="\t", skiprows=hdr+1, names=cols, comment="%")
    # alias new strain columns (eKen* Kenesei small-strain, written in µε)
    # → legacy E* (raw dimensionless). _mean_abs_strain_ue multiplies by 1e6
    # to report µε, so dividing here keeps the final units right.
    for _new, _old in {"eKen11":"E11","eKen22":"E22","eKen33":"E33",
                       "eKen23":"E23","eKen13":"E13","eKen12":"E12"}.items():
        if _new in df.columns and _old not in df.columns:
            df[_old] = df[_new] / 1.0e6   # µε → dimensionless
    return n, df

def _mean_abs_strain_ue(df):
    ecols = [c for c in ("E11","E22","E33","E12","E13","E23") if c in df.columns]
    if not ecols:
        return float("nan")
    return float(np.mean(np.abs(df[ecols].values)) * 1e6)

n1, g1 = _load_grains(RECON_OUT / "LayerNr_1")
n2, g2 = _load_grains(RECON_OUT2 / "LayerNr_1")
print(f"grains:             tx=0 -> {n1}      tx refined -> {n2}")
print(f"mean|E_ij| (ue):    {_mean_abs_strain_ue(g1):.0f}  ->  {_mean_abs_strain_ue(g2):.0f}")
print(f"median Confidence:  {g1['Confidence'].median():.4f}  ->  {g2['Confidence'].median():.4f}")
if abs(tx_ref) < 1e-4:
    print(f"\n(tx refined to {tx_ref:+.5f} deg ~ 0 on this dataset -> pass 2 == pass 1; "
          "tx's effect here is sub-pixel. The absolute strain is handled by the d0 "
          "recovery in section 6 regardless.)")

### Note — absolute strain, Lsd, and why d0 recovery is the right move

The per-grain **deviatoric** strain (shears + anisotropy) is pinned directly by the
diffraction data and is correct as-is. The **isotropic** part, however, rides on two
things that are *not* real strain:

1. the assumed reference `a = 3.6 Å` (the true strain-free `a0` is larger), and
2. the absolute `Lsd` — `d_obs ∝ Lsd`, so a 0.1 % Lsd offset shows up as ~1000 µε of
   uniform hydrostatic 'strain'.

Powder (LaB6) calibration does not pin `Lsd` to better than ~0.1 % (and is structurally
blind to `tx`). Rather than chase `Lsd` with a grain-based geometry refit, we recover
`a0` from the grains themselves (§6). For a cubic free-standing polycrystal the
strain-free reference is exactly the volume-weighted hydrostatic mean, so **recovering
`a0` absorbs both the reference offset and the residual `Lsd` error in one step** — the
absolute strain becomes insensitive to the powder-`Lsd` uncertainty. After d0 the mean
strain drops ~3× to the deviatoric floor (~200-300 µε), matching the legacy pipeline's
own floor. This is why the two-pass above refines `tx` only.


## 6. Fit the strain-free reference `d0` (midas-stress)

Powder calibration fixes the **wavelength**, but not the sample's **strain-free
lattice parameter** `a0`. Using the nominal value (3.6 Å for Ni) leaves a
spurious *hydrostatic* offset in every grain's strain (here ≈ +800 µε — Ni-ht is
thermally expanded vs nominal). For a **cubic, free-standing** polycrystal,
macroscopic equilibrium (mean stress = 0) recovers `a0` exactly from the
volume-weighted hydrostatic strain — **no single-crystal stiffness needed**
(`midas_stress.recover_d0_cubic_free_standing`).

> The *deviatoric* part of the strain is unaffected by `d0`; a large lab-frame
> deviatoric (one axis ≫ others) usually indicates a residual `Lsd`/wavelength
> error rather than real strain — worth checking separately.

In [ ]:
import numpy as np, pandas as pd
from midas_stress import recover_d0_cubic_free_standing

A0_NOMINAL = 3.6                       # nominal strain-free a (Å) for your phase
gfile = RECON_OUT / "LayerNr_1" / "Grains.csv"
lines = gfile.read_text().splitlines()
hdr = next(i for i, l in enumerate(lines) if l.startswith("%ID") or l.startswith("%GrainID"))
cols = lines[hdr].lstrip("%").split()
g = pd.read_csv(gfile, sep="\t", skiprows=hdr + 1, names=cols, comment="%")
# Alias new strain columns. midas-process-grains writes `eKen*` (Kenesei
# small strain) and `eFab*` (Fable-Beaudoin) in MICROSTRAIN (µε) — see
# io/csv.py:57-58. The legacy C ProcessGrains wrote `E11..E33` as raw
# dimensionless strain (ratio, ~1e-3). Divide by 1e6 so the rest of this
# cell — which expects raw strain in `A0 * (1 + ε)` — works unchanged.
_strain_map = {"eKen11": "E11", "eKen22": "E22", "eKen33": "E33",
               "eKen23": "E23", "eKen13": "E13", "eKen12": "E12"}
for _new, _old in _strain_map.items():
    if _new in g.columns and _old not in g.columns:
        g[_old] = g[_new] / 1.0e6   # µε → dimensionless

sel = g.Confidence >= 0.85             # use well-fit grains for the reference
n = int(sel.sum())
# Per-grain lattice from the (lab) strain — the hydrostatic trace is frame-
# invariant, which is all the cubic free-standing recovery uses.
lat = np.column_stack([A0_NOMINAL * (1 + g.E11[sel]), A0_NOMINAL * (1 + g.E22[sel]),
                       A0_NOMINAL * (1 + g.E33[sel])] + [np.full(n, 90.0)] * 3)
res = recover_d0_cubic_free_standing(
    lat, np.array([A0_NOMINAL] * 3 + [90.0] * 3),
    volumes=g.GrainRadius[sel].values ** 3,
    confidences=g.Confidence[sel].values, min_confidence=0.85)

a0 = float(res["reference_recovered"][0])
print(f"strain-free a0:  {A0_NOMINAL:.4f} → {a0:.5f} Å   "
      f"(hydrostatic ε_iso = {res['eps_iso']*1e6:+.0f} µε, {res['n_grains_used']} grains)")

# Re-center every grain's strain on the recovered d0 (remove the isotropic offset).
eps = res["eps_iso"]
for c in ("E11", "E22", "E33"):
    g[c + "_d0"] = g[c] - eps
vm0 = np.sqrt((g.loc[sel, ["E11", "E22", "E33"]].values ** 2).sum(1)) * 1e6
vm1 = np.sqrt((g.loc[sel, ["E11_d0", "E22_d0", "E33_d0"]].values ** 2).sum(1)) * 1e6
print(f"von Mises (normal) median:  before d0 = {np.median(vm0):.0f} µε  "
      f"→  after d0 = {np.median(vm1):.0f} µε")

## 7. Per-grain trust scoring (refiner-anchored UQ)

For each grain the refiner already produced a per-spot prediction (in
`FitBest.bin`). `midas-uq` reads those predictions directly — no
forward-model re-prediction, no convention drift — and bootstraps the
spot population to give a per-grain trust score:

* `pos_med_um` — median position residual on the detector
* `ome_med_deg` — median ω residual
* `angle_med_deg` — median g-vector misorientation (lab frame)
* `pos_med_boot_std_um` — bootstrap stability of `pos_med`

Thresholds are **derived from the detector geometry**, not hand-picked.
A grain is "trustworthy" when its median residual is within a couple of
pixels in space, ≤ half an ω frame in rotation, and the median is
stable to a fraction of a pixel under bootstrap resampling.

In [ ]:
# Score every grain — reads FitBest predictions, bootstraps spot residuals.
import os, time
import numpy as np
from midas_fit_grain import fitbest_from_array
from midas_uq import trust_score_anchored

layer_dir = RECON_OUT / "LayerNr_1"
gfile = layer_dir / "Grains.csv"
fb_path = layer_dir / "Output" / "FitBest.bin"

# Parse Grains.csv header so we don't depend on column order
lines = gfile.read_text().splitlines()
hdr_i = next(i for i, l in enumerate(lines) if l.startswith("%ID") or l.startswith("%GrainID"))
hdr = lines[hdr_i].lstrip("%").split()
gdata = np.array([[float(x) for x in l.split()]
                  for l in lines[hdr_i+1:] if l.strip() and not l.startswith("%")])
col = {n: i for i, n in enumerate(hdr)}
grain_ids = gdata[:, col["ID" if "ID" in col else "GrainID"]].astype(int)
gX = gdata[:, col["X"]]; gY = gdata[:, col["Y"]]; gZ = gdata[:, col["Z"]]
gConf = gdata[:, col["Confidence"]]

# FitBest is a (n_seeds, 5000, 22) float64 memmap — n_seeds comes from the
# *file size* (not from grain_ids.max() — IDs are not contiguous indices
# into FitBest in the v0.4.x refiner output).
ROW_BYTES = 5000 * 22 * 8
N_SEEDS = os.path.getsize(fb_path) // ROW_BYTES
fb = np.memmap(fb_path, dtype=np.float64, mode="r", shape=(N_SEEDS, 5000, 22))

print(f"scoring {len(grain_ids)} grains against {N_SEEDS} seed rows (n_boot=100)...", flush=True)
t0 = time.time()
trust = []
for k, gid in enumerate(grain_ids):
    rep = int(np.clip(gid - 1, 0, N_SEEDS - 1))
    view = fitbest_from_array(fb, rep)
    if view.n_spots == 0:
        trust.append(None); continue
    trust.append(trust_score_anchored(view, n_boot=100))
    if (k + 1) % 1000 == 0:
        print(f"  {k+1}/{len(grain_ids)}", flush=True)
print(f"done in {time.time() - t0:.1f} s")

In [ ]:
# Build the trust mask. The refiner-anchored UQ score is RELATIVE — it ranks
# grains within the population (their median spot residual under the refiner's
# own predictions). Geometry-derived "must be < 1 pixel" thresholds depend on
# the refiner's absolute accuracy: legacy FitPosOrStrainsOMP converges to
# ~0.8 px median residual, the c-omp midas_fitgrain converges to ~3 px.
# Either way, the trust signal is the *upper tail* — grains whose residuals
# are well above the population median (and whose bootstrap-medians are
# unstable). We use percentile thresholds (per-dataset) so the mask
# adapts to the refiner that produced this Grains.csv.
def _arr(getter):
    return np.array([getter(t) if t is not None else np.nan for t in trust])
pos_med  = _arr(lambda t: t.pos_med_um)
pos_p95  = _arr(lambda t: t.pos_p95_um)
ome_med  = _arr(lambda t: t.ome_med_deg)
ome_p95  = _arr(lambda t: t.ome_p95_deg)
ang_med  = _arr(lambda t: t.angle_med_deg)
pos_bstd = _arr(lambda t: t.pos_med_bootstrap_std_um)
n_spots  = _arr(lambda t: float(t.n_spots))

ok = ~np.isnan(pos_med)
# Data-driven thresholds: keep grains whose pos_med, ome_med, and bootstrap
# stability are all in the LOWER HALF of the dataset distribution. This is
# what "trustworthy = relative" means for a refiner-anchored score.
TH_POS  = float(np.median(pos_med[ok]))
TH_OME  = float(np.median(ome_med[ok]))
TH_BSTD = float(np.median(pos_bstd[ok]))
print(f"per-dataset thresholds (median of population):")
print(f"  pos_med      < {TH_POS:6.1f} µm")
print(f"  ome_med      < {TH_OME:6.4f} °")
print(f"  pos_boot_std < {TH_BSTD:6.1f} µm")

trust_mask = ok & (pos_med < TH_POS) & (ome_med < TH_OME) & (pos_bstd < TH_BSTD)
print(f"\ntrustworthy (all 3 below median): {trust_mask.sum()}/{ok.sum()} "
      f"({100 * trust_mask.sum() / max(ok.sum(), 1):.1f} %)")

In [ ]:
# Per-grain trust table — drop alongside Grains.csv
import csv
trust_csv = layer_dir / "GrainTrust.csv"
with open(trust_csv, "w", newline="") as fp:
    w = csv.writer(fp, delimiter="\t")
    w.writerow(["GrainID", "n_spots",
                "pos_med_um", "pos_p95_um",
                "ome_med_deg", "ome_p95_deg", "angle_med_deg",
                "pos_med_boot_std_um", "ome_med_boot_std_deg",
                "X", "Y", "Z", "Confidence", "Trust"])
    for i, gid in enumerate(grain_ids):
        if trust[i] is None:
            w.writerow([int(gid), 0] + ["nan"] * 7
                       + [f"{gX[i]:.3f}", f"{gY[i]:.3f}", f"{gZ[i]:.3f}",
                          f"{gConf[i]:.4f}", 0])
            continue
        t = trust[i]
        w.writerow([int(gid), t.n_spots,
                    f"{t.pos_med_um:.3f}", f"{t.pos_p95_um:.3f}",
                    f"{t.ome_med_deg:.5f}", f"{t.ome_p95_deg:.5f}",
                    f"{t.angle_med_deg:.5f}",
                    f"{t.pos_med_bootstrap_std_um:.3f}",
                    f"{t.ome_med_bootstrap_std_deg:.6f}",
                    f"{gX[i]:.3f}", f"{gY[i]:.3f}", f"{gZ[i]:.3f}",
                    f"{gConf[i]:.4f}", int(trust_mask[i])])
print(f"wrote {trust_csv}")


### Trust distribution — six residual histograms

Dashed lines show the median of each distribution; dotted red lines mark
the geometry-derived thresholds. A clean run sits well below all three
thresholds; a long tail to the right is the signature of grains whose
spot association is partly wrong (overlap, false-twin, or a real
ambiguity in the indexer).

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
panels = [
    ("pos_med (µm)",         pos_med[ok],  TH_POS,  "tab:blue"),
    ("pos_p95 (µm)",         pos_p95[ok],  None,    "tab:cyan"),
    ("ome_med (°)",          ome_med[ok],  TH_OME,  "tab:orange"),
    ("ome_p95 (°)",          ome_p95[ok],  None,    "tab:red"),
    ("angle_med (°)",        ang_med[ok],  None,    "tab:purple"),
    ("pos_med boot_std (µm)", pos_bstd[ok], TH_BSTD, "tab:green"),
]
for k, (name, vals, thr, c) in enumerate(panels):
    a = ax[k // 3, k % 3]; hi = float(np.percentile(vals, 99))
    a.hist(vals, bins=80, color=c, alpha=0.75, range=(0, max(hi, 1e-6)))
    a.axvline(float(np.median(vals)), color="k", ls="--", lw=1, alpha=0.6,
              label=f"median = {np.median(vals):.3g}")
    if thr is not None:
        a.axvline(thr, color="r", ls=":", lw=1.5, label=f"threshold = {thr:.3g}")
    a.set_xlabel(name); a.set_ylabel("# grains"); a.legend(fontsize=8)
fig.suptitle(f"Refiner-anchored UQ — {ok.sum()} grains, "
             f"trustworthy = {trust_mask.sum()} "
             f"({100 * trust_mask.sum() / max(ok.sum(), 1):.0f} %)", fontsize=13)
plt.tight_layout(); plt.show()


### Spatial trust map — where do the bad grains live?

Two diagnostic panels colored by the trust metrics, plus an explicit
trust mask. Low-trust grains often cluster at the beam edge (low
illumination, partial spots) or sit on top of well-fit neighbours
(ambiguous indexing). The trust mask is the audited "use these"
population for downstream analysis.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5.6))
vmax_p = float(np.percentile(pos_med[ok], 95))
vmax_o = float(np.percentile(ome_med[ok], 95))

s0 = ax[0].scatter(gY[ok], gZ[ok], c=pos_med[ok], s=8, cmap="viridis_r",
                   vmin=0, vmax=vmax_p)
ax[0].set_xlabel("Y (µm)"); ax[0].set_ylabel("Z (µm)")
ax[0].set_aspect("equal"); ax[0].set_title("UQ pos_med (Y-Z)")
plt.colorbar(s0, ax=ax[0], label="µm")

s1 = ax[1].scatter(gY[ok], gZ[ok], c=ome_med[ok], s=8, cmap="plasma",
                   vmin=0, vmax=vmax_o)
ax[1].set_xlabel("Y (µm)"); ax[1].set_ylabel("Z (µm)")
ax[1].set_aspect("equal"); ax[1].set_title("UQ ome_med (Y-Z)")
plt.colorbar(s1, ax=ax[1], label="°")

ax[2].scatter(gY[ok & ~trust_mask], gZ[ok & ~trust_mask], s=4, c="lightgrey",
              alpha=0.5, label=f"low trust ({(ok & ~trust_mask).sum()})")
ax[2].scatter(gY[trust_mask], gZ[trust_mask], s=8, c="tab:green",
              alpha=0.7, label=f"trustworthy ({trust_mask.sum()})")
ax[2].set_xlabel("Y (µm)"); ax[2].set_ylabel("Z (µm)"); ax[2].set_aspect("equal")
ax[2].set_title(f"trust mask "
                f"(pos<{TH_POS:.0f} & ω<{TH_OME:.2f} & boot<{TH_BSTD:.0f})")
ax[2].legend(fontsize=9)
plt.tight_layout(); plt.show()


### Trust vs. confidence — the two are *not* the same

`Confidence` is the C refiner's matched-spot fraction. `pos_med` is the
geometric residual on the matched spots themselves. They correlate but
disagree on a non-trivial subpopulation: high-confidence grains can still
carry a poorly-fit spot stack, and low-confidence grains with few clean
spots can be highly trustworthy. Filter on **both** when you want a
high-purity grain set for downstream property estimation.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(gConf[ok], pos_med[ok], s=5, alpha=0.4, c="tab:blue")
ax[0].axhline(TH_POS, color="r", ls=":", lw=1, label=f"pos threshold = {TH_POS:.0f} µm")
ax[0].set_xlabel("Confidence (refiner)"); ax[0].set_ylabel("UQ pos_med (µm)")
ax[0].set_title("trust vs. confidence"); ax[0].legend(fontsize=9)

ax[1].scatter(gConf[ok], pos_bstd[ok], s=5, alpha=0.4, c="tab:green")
ax[1].axhline(TH_BSTD, color="r", ls=":", lw=1, label=f"stability threshold = {TH_BSTD:.0f} µm")
ax[1].set_xlabel("Confidence (refiner)"); ax[1].set_ylabel("UQ bootstrap_std (µm)")
ax[1].set_title("stability vs. confidence"); ax[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

# Re-fit d0 on the trustworthy subset only — apples-to-apples with §6
sel_trust = trust_mask & (gConf >= 0.85)
print(f"trustworthy + confident grains used for d0:  {int(sel_trust.sum())} / {ok.sum()}")


## 8. v4 pipeline (Stage 8.5 budget drop + force-keep distinct + orphan reclaim)

`midas-process-grains` v0.4.4 adds a second, complementary trust system on top of the c-omp
reconstruction in §3 and the refiner-anchored UQ in §7:

* **Stage 6 (NNLS)** — re-attributes intensity between split-product grains.
* **Stage 7** — per-grain (σ_X, σ_Y, σ_Z) from Hessian inversion.
* **Stage 8** — Kenesei bounded per-grain strain (Voigt 6).
* **Stage 8.5** — drops candidates by quality until ΣV_kept ≤ V_sample.
* **Stage 8.5b** — force-keeps candidates whose nearest kept grain is beyond peak-search
  resolution in misorientation AND beyond 3σ in position.
* **Stage 8.5c** — reclaims dropped grains that uniquely cover orphan spots.

Emits a `GrainsV4.csv` leaf with `trust_tier_sigma_aware` (0/1/2 = bronze/silver/gold),
`drop_by_budget`, per-grain σ_X/Y/Z, hkl_coverage, strain ε_11..ε_23, etc.

Tell the pipeline the actual illuminated sample volume in paramstest as `Vsample`
(thin foils: set `DiscModel 1; DiscArea <V_um3>;` for the lateral-disc R interpretation).


In [ ]:
from midas_process_grains.v4_pipeline import run_v4_pipeline
import midas_process_grains as _mpg
print(f"midas_process_grains v{_mpg.__version__}")

# Tell v4 the *physical* illuminated sample volume (300×300×200 µm³ for this
# Ni sample). v4 will scale the per-grain radii by (V_truth / V_gauge_legacy)^(1/3)
# so the leaf reports physical R / V — and Stage 8.5 will use this V_truth as
# the budget for the per-grain + family intensity-conservation drops.
# Requires midas-process-grains >= 0.4.6 (handles the c-omp meanRadius=1
# placeholder correctly; on older versions the radii are shrunk 4× by the
# correction).
ps = RECON_OUT / "LayerNr_1" / "paramstest.txt"
if "Vsample" not in ps.read_text():
    V_SAMPLE_UM3 = 1.8e7   # ← edit for your sample (300×300×200 µm³ here)
    with ps.open("a") as f:
        f.write(f"Vsample {V_SAMPLE_UM3};\n")

paths = run_v4_pipeline(
    layer_dir=RECON_OUT / "LayerNr_1",
    out_dir=RECON_OUT / "LayerNr_1" / "v4_v045_out",
    merge_primitive="forward_predict",
    trust_scheme="sigma_aware",
    compute_position_sigma=True,
    position_sigma_max_grains=None,        # full coverage; sub-sample for large datasets
    compute_strain=True,
    # Stage 8.5 — intensity-conservation budget (per-grain + family-max)
    drop_budget_tolerance=1.0,
    # Stage 8.5b — force-keep distinct (Path 2)
    force_keep_distinct_enabled=True,
    force_keep_distinct_misori_deg=1.0,     # 0.5 = peak-search resolution; 1.0 = conservative
    force_keep_distinct_sigma=3.0,
    # Stage 8.5c — orphan-greedy reclaim (Path 3); v0.4.5 default frac=0.5
    orphan_reclaim_enabled=True,
    orphan_reclaim_min_unique_spots=5,
    orphan_reclaim_min_unique_fraction=0.5,
)
print(paths)

### Read the v4 leaf and produce the per-grain trust map

Default user filter: `trust_tier_sigma_aware >= 1` keeps gold + silver only.
Strict intensity-conservation: filter `drop_by_budget == 0` (= what the pipeline already decided).


In [ ]:
import pandas as pd, numpy as np, math
import matplotlib.pyplot as plt

leaf = pd.read_csv(paths["leaf"], sep="\t")
kept = leaf["drop_by_budget"] == 0
print(f"N total = {len(leaf):,}   kept = {kept.sum():,}   dropped = {(~kept).sum():,}")
for t in (2, 1, 0):
    nk = ((leaf["trust_tier_sigma_aware"] == t) & kept).sum()
    nt = (leaf["trust_tier_sigma_aware"] == t).sum()
    print(f"  tier {t} ({'gold' if t==2 else 'silver' if t==1 else 'bronze'}): {nk:,}/{nt:,} kept")

# X-Y grain map colored by trust tier
fig, ax = plt.subplots(figsize=(8, 8))
tier_colors = {0: "#888888", 1: "#C0C0C0", 2: "#FFD700"}
for t, c in tier_colors.items():
    sub = leaf[(leaf["trust_tier_sigma_aware"] == t) & kept]
    ax.scatter(sub["X"], sub["Y"], c=c, s=10, alpha=0.7, edgecolor="k", linewidth=0.2,
                label=f"tier {t} (n={len(sub):,})")
ax.set_xlabel("X (µm)"); ax.set_ylabel("Y (µm)"); ax.set_aspect("equal")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("v4 grain map — colored by trust tier (kept population only)")
plt.show()


### Cross-reference §7 (refiner-anchored UQ) and §8 (Hessian σ + budget) trust

Both per-grain trust metrics are produced from the same diffraction data via
independent routes:

* §7 `pos_med_um`, `ome_med_deg` — residuals between refined OM/position and FitBest predictions.
* §8 `sigma_X/Y/Z_um`, `trust_tier_sigma_aware` — Hessian inversion + budget filter.

Agreement between them is a robust sanity check. Disagreement flags a grain whose claim of
good Confidence is undermined by one of the two checks.
